# 🎬 VAJRA 2.0 — AI Media Studio

**7 modules:** Video Edit · Voice Generation · Text→Image (RealVisXL) · Face Swap · Text→Video (LTX 2.3 / Wan2.2-I2V) · Image Edit (Qwen-Image-Edit-2509) · Media Studio.

**How to run:** `Runtime → Change runtime type → GPU` (**A100** recommended), then run the cells **top to bottom**.

### Build only what you need

Steps 1–5 are always required. **Steps 6 and 7 are selective** — set the flags in those cells for the modules you actually want to use. Building every environment takes many minutes; building one module's takes a fraction of that.

| Module | Step 6 environments | Step 7 weight groups |
|---|---|---|
| **Video Edit** | `VOICE` + `LIPSYNC` (+ `SEPARATE`, recommended) | `voice`, `lipsync` |
| **Voice Generation** | `VIITOR` | `viitor` |
| **Image Generation** | *none* | *none* (fetched on first use) |
| **Face Swap** | *none* | `faceswap` |
| **Media Studio** | *none* | *none* |
| **Text → Video** | `LTX2` and/or `WAN` | *none* (fetched on first use) |
| **Image Edit** | `QWEN` | *none* (fetched on first use) |

- Set **`USE_DRIVE = True`** in Step 2 to cache weights on Drive so nothing re-downloads next session.
- **No tokens needed** for any default module. Wan2.2-I2V and LTX 2.3 *might* need one depending on their access settings — see Step 4.


## ✅ **Step 1 — Check the GPU**
Confirms an A100/L4 is attached. If this errors, set `Runtime → Change runtime type → GPU` first.

In [ ]:
# 1. Check GPU
!nvidia-smi

## 💾 **Step 2 — (Optional) Cache model weights on Drive**
Set `USE_DRIVE = True` to cache **all model weights** on your Drive — both Step 7's downloads and anything a module fetches on first use (SDXL, LTX 2.3, Wan2.2, Qwen-Image-Edit, …). Nothing re-downloads on your next session. **Environments stay local** — Drive can't execute a venv's binaries, so Step 6 always builds them on local disk.

**Storage heads-up:** the complete model set exceeds 100 GB. Caching only grows as you actually use modules, but if you plan to exercise the heavy video/editing modules regularly, make sure you have room. Otherwise leave this `False` and accept re-downloading each session.


In [ ]:
# 2. (Optional) cache MODEL WEIGHTS on Drive to skip re-downloading next session.
#    This covers cell 7's explicit downloads AND anything fetched on first use
#    (SDXL/LTX/Wan2.2/Qwen-Image-Edit, ...) -- see core/config.py's HF_HUB_CACHE.
#    NOTE: venvs are deliberately NOT put on Drive — Google Drive can't execute a
#    venv's python ("bad interpreter: Permission denied"), so they always build
#    locally on /content.
USE_DRIVE = False  # set True to cache model weights on your Drive
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['VAJRA_MODELS'] = '/content/drive/MyDrive/vajra/models'
    os.makedirs(os.environ['VAJRA_MODELS'], exist_ok=True)
    os.environ.pop('VAJRA_VENVS', None)   # keep venvs local (Drive can't run them)

## 📥 **Step 3 — Get the project**
Clones the repo from GitHub into `/content`. Public repo works as-is; for a private repo add a `GITHUB_TOKEN` Colab secret (🔑 icon).

In [ ]:
# 3. Get the project from GitHub (public repo works as-is).
import os
REPO_USER = 'raunakmalix-ctrl'
REPO_NAME = 'VAJRA-v1.1'
BRANCH    = 'vajra-2.0'   # the VAJRA 2.0 work lives here until it merges to main
ROOT = f'/content/{REPO_NAME}'

token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass

auth = f'{token}@' if token else ''
REPO_URL = f'https://{auth}github.com/{REPO_USER}/{REPO_NAME}.git'
# The branch is pinned explicitly: cloning the default branch would silently
# fetch main, which has none of the VAJRA 2.0 work and an older setup script.
if os.path.isdir(ROOT):
    # Already cloned this session -- fetch and fast-forward, so re-running this
    # cell picks up fixes instead of silently keeping a stale checkout.
    !git -C $ROOT fetch origin $BRANCH
    !git -C $ROOT checkout $BRANCH
    !git -C $ROOT pull --ff-only
else:
    !git clone --branch $BRANCH $REPO_URL $ROOT
!git -C $ROOT log --oneline -1
os.environ['VAJRA_ROOT'] = ROOT
%cd $ROOT

## 🔑 **Step 4 — (Optional) Hugging Face token**

Every default module is confirmed **open** — SDXL Realistic, Qwen-Image-Edit, LatentSync, XTTS-v2. You only need a token if **Wan2.2-I2V** or **LTX 2.3** (Text→Video's engines) turn out to be gated; this wasn't confirmed either way when they were integrated.

**If a video generation fails with an access/gated error:**
1. Free account at **huggingface.co**.
2. Token: **huggingface.co/settings/tokens → New token → type *Read* → Create →** copy the `hf_…` value.
3. Visit the model page (**huggingface.co/Wan-AI/Wan2.2-I2V-A14B-Diffusers** or **huggingface.co/diffusers/LTX-2.3-Diffusers**) while logged in and accept its licence if prompted.
4. Paste the token into the cell below.

*Everything else works with this left blank.*


In [ ]:
# 4. Hugging Face token -- OPTIONAL. Only needed if Wan2.2-I2V or LTX 2.3
#    (Text -> Video's engines) turn out to be gated.
#    Every default tab (SDXL, Qwen-Image-Edit, LatentSync) works with this blank.
import os
os.environ['HF_TOKEN'] = ''  # <- paste your hf_... token here, or leave empty


## ⚙️ **Step 5 — Principal runtime** *(always required)*
Installs ffmpeg, clones the third-party model repositories (Wav2Lip, LatentSync, CodeFormer) and pip-installs the main environment. A few minutes. **Re-run this after a `git pull` that adds new repositories.**


In [ ]:
# 5. Main env: ffmpeg + clone model repos + pip install
#    On a runtime whose python is 3.13 or newer this builds venv_main on
#    python3.10 first (the stack's numpy<2 floor has no 3.13 wheels), which
#    adds a one-off torch download. First run ~10 min, later runs ~2.
!bash setup/install_main.sh


## 🧩 **Step 6 — Build ONLY the environments you need** *(selective)*

Each module's dependency stack is mutually incompatible with the others, so each lives in its own isolated environment. **Set the flags below for the modules you intend to use.** Selected targets build **concurrently**; unselected ones cost nothing.

Re-running is safe: an environment that already exists is skipped, so you can come back later and add another module without rebuilding what you have.

> Testing **Video Edit** first? Leave `VOICE`, `LIPSYNC` and `SEPARATE` on and everything else off.
> Testing **Image Generation**, **Face Swap** or **Media Studio**? Turn *all* flags off — those run in the principal runtime.

**About `SEPARATE`** — Demucs splits the audio into a speech stem and a background stem. Video Edit then edits *only* the speech and re-lays the untouched background over the result, so room tone, music and traffic play continuously across the edit because they were never regenerated. It is the strongest single realism measure in the pipeline. Skipping it still works — the edit just has more to disguise.


In [ ]:
# 6. Build ONLY the isolated environments the modules you want actually need.
#
#    Module          Needs
#    --------------  -------------------------------------
#    Video Edit    VOICE + LIPSYNC  (+ SEPARATE, optional but recommended)
#                    (+ VIITOR for tier A infill - best quality, English)
#    Voice Edit      VIITOR
#    Image Generation   (nothing)
#    Face Swap       (nothing)
#    Media Studio    (nothing)
#    Text -> Video   LTX2  and/or  WAN
#    Image Edit      QWEN

VOICE    = True    # Video Edit  - voice cloning (XTTS-v2)
LIPSYNC  = True    # Video Edit  - lip re-sync (LatentSync + Wav2Lip)
SEPARATE = False   # Video Edit  - speech/background separation (Demucs v4,
                   #                 only needed for music/noisy backgrounds)
VIITOR   = False   # Voice Edit    - ViiTorVoice-NAR tier A infill (heavy, English)
WAN      = False   # Text -> Video - Wan2.2-I2V motion video (heavy)
QWEN     = False   # Image Edit    - Qwen-Image-Edit-2509 (heavy)
LTX2     = False   # Text -> Video - LTX-2.3, video + synced audio (heavy)

_targets = [n for n, on in [('voice', VOICE), ('lipsync', LIPSYNC),
                            ('demucs', SEPARATE), ('viitor', VIITOR),
                            ('wan', WAN),
                            ('qwen', QWEN), ('ltx2', LTX2)] if on]
if _targets:
    print('Building:', ' '.join(_targets))
    !bash setup/make_venvs.sh {' '.join(_targets)}
else:
    print('No environments selected. Text->Image / Face Swap / Media Studio '
          'need none — they run in the principal runtime.')
    !bash setup/make_venvs.sh

## ⬇️ **Step 7 — Download ONLY the weights you need** *(selective)*

Anything you skip here is fetched automatically the first time you use that module, so skipping only defers the download — it never breaks a module.

| Group | Weights | Needed by |
|---|---|---|
| `voice` | XTTS-v2 | Video Edit |
| `lipsync` | LatentSync 1.5 · Wav2Lip GAN | Video Edit |
| `faceswap` | inswapper_128 · GFPGAN v1.4 | Face Swap |

Idempotent — safe to re-run. Use `['all']` to fetch everything.


In [ ]:
# 7. Download ONLY the weight groups you need.
#    Groups: 'voice', 'lipsync', 'faceswap', 'viitor'  (or ['all'], or [] to skip)
#    'viitor' is a multi-GB download - only add it if VIITOR was built above.
#    You can also name a module: 'relip', 'faceswap', 'txt2img', ...
DOWNLOAD = ['voice', 'lipsync']    # default: what Video Edit needs

if DOWNLOAD:
    !python setup/download_models.py {' '.join(DOWNLOAD)}
else:
    print('Skipped — every module fetches its weights on first use.')


## 🚀 **Step 8 — Launch**
Starts the app and prints a public **`*.gradio.live`** link. Keep this cell running while you use the studio. Modules whose environment you didn't build will report a clear message telling you which Step 6 flag to enable.


In [ ]:
# 8. Launch the app — click the public *.gradio.live link in the output
os.environ['VAJRA_SHARE'] = '1'

# Use venv_main when install_main.sh had to build one (see cell 5); the
# ambient interpreter cannot import this stack in that case.
PY = 'venv_main/bin/python' if os.path.exists('venv_main/bin/python') else 'python'
print('launching with', PY)
!{PY} app.py
